<a href="https://colab.research.google.com/github/mdkamrulhasan/gvsu_machine_learning/blob/main/notebooks/supervised/Random_forest_sm_baseline_kaggle_upload.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Random Forest Model for Predicting `sm_surface`

This notebook demonstrates how to:
1. Load time-series training data (`training_data.csv`).
2. Create calendar features (year, month, week of year).
3. Create lag features from `sm_surface`.
4. Train a Random Forest regression model.
5. Generate predictions for future dates (**Oct 1, 2024 → Jan 01, 2025**).


In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from datetime import timedelta
import os

In [6]:

from google.colab import drive
drive.mount('/content/drive')


# Final data (Test starts at Oct 1, 2025)
base_path = 'drive/MyDrive/data/time-series/W26-Gaylord_MI-Test-start:2024-10-01'

Mounted at /content/drive


## Load and Process Data

In [7]:
# Number of lags to consider
NB_LAGS = 3

In [8]:
def apply_feature_engineering(df, nb_lags=3, test_data=False):
    """Feature engineering"""

    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)

    if not test_data:
      for i in range(1, nb_lags+1):
        df['lag{}'.format(i)] = df['sm_surface'].shift(i)
    else:
      for i in range(1, nb_lags+1):
        # Placeholder data (Not real)
        df['lag{}'.format(i)] = df_train.sm_surface.mean()

    return df.fillna(df.mean(numeric_only=True)).reset_index(drop=True)

### Load and Preprocess Training Data

In [9]:
# Training data
df_train = pd.read_csv(os.path.join(base_path, 'training_data.csv'))
df_train['date'] = pd.to_datetime(df_train['date'])
df_train = df_train.sort_values('date').reset_index(drop=True)
df_train = apply_feature_engineering(df_train, nb_lags=NB_LAGS)
df_train.head()

,date,sm_surface,year,month,weekofyear,lag1,lag2,lag3
0,2023-01-01,0.156469,2023,1,52,0.149305,0.149367,0.149425
1,2023-01-02,0.155497,2023,1,1,0.156469,0.149367,0.149425
2,2023-01-03,0.155756,2023,1,1,0.155497,0.156469,0.149425
3,2023-01-04,0.154115,2023,1,1,0.155756,0.155497,0.156469
4,2023-01-05,0.145088,2023,1,1,0.154115,0.155756,0.155497


### Load and Preprocess Test Data

In [10]:
# Test data
df_test = pd.read_csv(os.path.join(base_path, 'test_data_X.csv'))
df_test['date'] = pd.to_datetime(df_test['date'])
df_test = df_test.sort_values('date').reset_index(drop=True)
df_test.head()

# Apply feature enginerring to test data
df_test = apply_feature_engineering(df_test, test_data=True, nb_lags=NB_LAGS)
df_test.head()

,date,year,month,weekofyear,lag1,lag2,lag3
0,2024-10-01,2024,10,40,0.149244,0.149244,0.149244
1,2024-10-02,2024,10,40,0.149244,0.149244,0.149244
2,2024-10-03,2024,10,40,0.149244,0.149244,0.149244
3,2024-10-04,2024,10,40,0.149244,0.149244,0.149244
4,2024-10-05,2024,10,40,0.149244,0.149244,0.149244


## Training Random Forest Model

In [11]:
## CAREFUL: lags must be at the end of the list
features = ['year','month','weekofyear'] + ['lag{}'.format(i+1) for i in range(NB_LAGS)]
target = 'sm_surface'

X = df_train[features].values
y = df_train[target].values

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X, y)
print("Model trained.")


Model trained.


# Predict per row so we produce the results in the exact sequence ncecessary for the kaggle challenge

In [12]:
def get_rolling_predictions(df_train, df_test):

  predictions = list(df_train.sm_surface.values[-NB_LAGS:])


  for index, row in df_test.iterrows():
    #print
    row_features = row[features].values.reshape(1, -1)

    # updating the lag features (must be aligned with the lag definitions)
    row_features_lags_updated = list(row_features[0])[:-NB_LAGS] + predictions[-NB_LAGS:]
    pred = model.predict(np.array(row_features_lags_updated).reshape(1, -1))[0]
    # updating the prediction tracking list
    predictions.append(pred)


  return predictions[NB_LAGS:]

predictions = get_rolling_predictions(df_train, df_test)

## Training Data and Predictions Visualization

In [33]:
trr_df = df_train[['date', 'sm_surface']].copy()
trr_df['is_train'] = True
tst_df = df_test[['date']].copy()
tst_df['sm_surface'] = predictions
tst_df['is_train'] = False
combined_df = pd.concat([trr_df, tst_df], ignore_index=True)
combined_df['date'] = pd.to_datetime(combined_df['date'])
combined_df = combined_df.sort_values('date')


In [36]:
import plotly.graph_objects as go
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=combined_df[combined_df.is_train==True]['date'],
    y=combined_df[combined_df.is_train==True]['sm_surface'],
    mode='lines',
    name='Training Moisture Level Data'
))

fig.add_trace(go.Scatter(
    x=combined_df[combined_df.is_train==False]['date'],
    y=combined_df[combined_df.is_train==False]['sm_surface'],
    mode='lines',
    name='Predited Moisture Level'
))

fig.update_layout(
    title='Ground Truth vs Predicted sm_surface',
    xaxis_title='Date',
    yaxis_title='sm_surface'
)

fig.show()


## Exporting Predictions for Kaggle Challenge Upload

In [ ]:
#Pack results in a pandas dataframe
test_predictions = pd.DataFrame({
    'rowId': range(0, len(predictions)),
    'label': predictions})
# Exporting resluts
test_predictions.to_csv(os.path.join(base_path, 'my_moisture_predictions_to_upload-to-kaggle.csv'), index=False)